# 🚀 Qwen2.5 QLoRA 4-Bit NormalFloat Training (Free Google Colab / T4 GPU)

This notebook demonstrates **QLoRA** (Quantized Low-Rank Adaptation) on `Qwen/Qwen2.5-0.5B` or `Qwen/Qwen2.5-7B` using **4-bit NormalFloat (NF4)** quantization with `bitsandbytes`.

In [ ]:
# 1. Install required packages
!pip install -q torch transformers peft datasets accelerate bitsandbytes rich

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
import json

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Configure 4-Bit NF4 Quantization (QLoRA)
model_id = "Qwen/Qwen2.5-0.5B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct" on GPU

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 3. Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
print("Model quantized into 4-bit NF4 successfully!")

In [ ]:
# 4. Attach LoRA Adapter
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Sample Dataset (ChatML Format)
SYSTEM_PROMPT = (
    "You are OpsPilot, an expert DevOps and Linux Systems Engineer. "
    "Always respond strictly in valid JSON with keys: command, explanation, safety_level, caution."
)

raw_data = [
    {
        "instruction": "Find all processes on port 8080 and show PIDs.",
        "response": '{"command": "lsof -i :8080 -t", "explanation": "Lists PIDs listening on 8080.", "safety_level": "SAFE", "caution": "None."}'
    },
    {
        "instruction": "Check disk space usage across all mounted filesystems.",
        "response": '{"command": "df -hT", "explanation": "Shows disk space and filesystem types in human-readable format.", "safety_level": "SAFE", "caution": "None."}'
    }
]

def format_chatml(inst, resp):
    return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{resp}<|im_end|>"

formatted = [format_chatml(d["instruction"], d["response"]) for d in raw_data]
tokenized = tokenizer(formatted, truncation=True, max_length=256, padding=False)
tokenized["labels"] = [ids[:] for ids in tokenized["input_ids"]]
dataset = Dataset.from_dict(tokenized)
print(f"Prepared {len(dataset)} training samples.")

In [ ]:
# 6. Train with QLoRA
training_args = TrainingArguments(
    output_dir="./qwen_qlora_colab",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    num_train_epochs=5,
    logging_steps=1,
    bf16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

trainer.train()
model.save_pretrained("./qwen_qlora_adapter")
tokenizer.save_pretrained("./qwen_qlora_adapter")
print("QLoRA Adapter successfully saved!")